# Code-Capacity Gaussian Memory Heatmaps

This notebook compares uniform interval sampling against truncated Gaussian sampling on the same support interval. The scripts in `scripts/` are the source of truth; this notebook is a lightweight companion for interactive inspection.


In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


In [2]:
def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "relay_bp").exists():
            return candidate
    raise FileNotFoundError("Could not locate the relay repo root.")


REPO_ROOT = find_repo_root(Path.cwd())
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from relay_bp.analysis import (
    best_heatmap_rows,
    default_export_code_paths,
    evaluate_matched_support_heatmaps,
    load_code_capacity_problem,
    matched_support_heatmap_rows,
    sample_random_error_cases,
    with_uniform_error_rate,
)

CODE_PATHS = default_export_code_paths(REPO_ROOT)
selected_code = "surface13"
p_random_error = 0.09
centers = np.linspace(-0.1, 0.3, 5)
widths = np.linspace(0.0, 0.6, 5)
heatmap_draws = 2
n_error_samples = 64
max_iter = 20
alpha = 1.0


c:\Users\User\Documents\projects-git\relay_mother_folder\relay\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
base_problem = load_code_capacity_problem(CODE_PATHS[selected_code])
problem = with_uniform_error_rate(base_problem, p_random_error)
cases = sample_random_error_cases(
    problem,
    num_samples=n_error_samples,
    error_rate=p_random_error,
    base_seed=0,
)
artifact = evaluate_matched_support_heatmaps(
    problem=problem,
    cases=cases,
    max_iter=max_iter,
    alpha=alpha,
    centers=centers.tolist(),
    widths=widths.tolist(),
    base_seed=0,
    random_draws_per_point=heatmap_draws,
    application_scope="all_bits",
    selection_mode="single_draw",
    logical_weight_penalty=4.0,
    convergence_penalty=1.0,
    iteration_penalty=0.05,
)
heatmap_table = pd.DataFrame(matched_support_heatmap_rows(artifact))
best_points = pd.DataFrame(best_heatmap_rows(heatmap_table.to_dict("records"), topk=8))
display(best_points)


,family,center,width,interval_low,interval_high,sigma,logical_success_rate,convergence_rate,exact_recovery_rate,mean_iterations,mean_logical_weight,loss_mean,trial_count
0,uniform_interval,0.1,0.60,-0.200,0.400,0.1500,0.523438,0.0,0.0,20.0,0.476562,2.95625,64
1,uniform_interval,0.2,0.60,-0.100,0.500,0.1500,0.515625,0.0,0.0,20.0,0.484375,2.98750,64
2,truncated_gaussian,0.3,0.60,0.000,0.600,0.1500,0.515625,0.0,0.0,20.0,0.484375,2.98750,64
3,truncated_gaussian,0.1,0.60,-0.200,0.400,0.1500,0.500000,0.0,0.0,20.0,0.500000,3.05000,64
4,truncated_gaussian,0.2,0.60,-0.100,0.500,0.1500,0.500000,0.0,0.0,20.0,0.500000,3.05000,64
5,uniform_interval,0.0,0.00,0.000,0.000,0.0000,0.500000,0.0,0.0,20.0,0.500000,3.05000,64
6,truncated_gaussian,0.0,0.00,0.000,0.000,0.0000,0.500000,0.0,0.0,20.0,0.500000,3.05000,64
7,uniform_interval,0.2,0.45,-0.025,0.425,0.1125,0.492188,0.0,0.0,20.0,0.507812,3.08125,64


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
for ax, family_name in zip(axes, ["uniform_interval", "truncated_gaussian"]):
    image = ax.imshow(
        artifact["families"][family_name]["logical_success_rate"],
        origin="lower",
        aspect="auto",
        cmap="viridis",
    )
    ax.set_title(family_name)
    ax.set_xlabel("center index")
    ax.set_ylabel("width index")
    fig.colorbar(image, ax=ax, shrink=0.8)
plt.show()


In [4]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "relay_bp").exists():
            return candidate
    raise FileNotFoundError("Could not locate the relay repo root.")


_repo_root = find_repo_root(Path.cwd())
_src_root = _repo_root / "src"
if str(_src_root) not in sys.path:
    sys.path.insert(0, str(_src_root))

from relay_bp.analysis import (
    default_export_code_paths,
    evaluate_matched_support_heatmaps,
    load_code_capacity_problem,
    matched_support_heatmap_rows,
    sample_random_error_cases,
    with_uniform_error_rate,
)

_code_paths = default_export_code_paths(_repo_root)
smoke_problem = with_uniform_error_rate(load_code_capacity_problem(_code_paths["surface13"]), 0.09)
smoke_cases = sample_random_error_cases(smoke_problem, num_samples=8, error_rate=0.09, base_seed=11)
smoke_artifact = evaluate_matched_support_heatmaps(
    problem=smoke_problem,
    cases=smoke_cases,
    max_iter=10,
    alpha=1.0,
    centers=[0.0, 0.2],
    widths=[0.0, 0.4],
    base_seed=5,
    random_draws_per_point=1,
    application_scope="all_bits",
    selection_mode="single_draw",
    logical_weight_penalty=4.0,
    convergence_penalty=1.0,
    iteration_penalty=0.05,
)
smoke_rows = matched_support_heatmap_rows(smoke_artifact)
assert len(smoke_rows) == 8
assert int(smoke_artifact["families"]["uniform_interval"]["trial_count"][0, 0]) == len(smoke_cases)
